# Create edgotype profiles

In this step, we make one edgotype profile per gene of interactors by alleles. Later, we will explore metrics that summarize overall perturbation for each allele (quantitative score like imaging / abundance)

In [1]:
import polars as pl

In [2]:
inputs_dir = "../1_inputs/sqY2H"
outputs_dir = "../3_outputs/sqY2H"

In [3]:
scores = pl.read_csv(f"{outputs_dir}/4_normalized_consensus_scores.csv")
scores

ad_orf_id,db_orf_id,db_mut_id,retest_batch,normalized_score,assay_id,seq_ad,seq_db,db_sequenced,db_sequence_confirmation_class,ad_symbol,symbol,aa_change,nt_change,ensembl_gene_id,collection,clinvar_clnsig_clean,gnomad_af,StarStatus,allele_0
i64,i64,i64,str,f64,i64,i64,i64,f64,f64,str,str,str,str,str,str,str,f64,f64,str
7045,6157,0,"""VUSAPWT1B2""",0.5,1,1,1,null,null,"""TNIP1""","""OPTN""",null,null,"""ENSG00000123240""",null,null,null,null,"""OPTN"""
100008545,4384,0,"""VUSAPWT1B2""",0.4375,1,1,1,null,null,"""MYC""","""MAX""",null,null,"""ENSG00000125952""",null,null,null,null,"""MAX"""
4623,9905,216952,"""VUSAPWT1B2""",0.375,1,1,1,1.0,1.0,"""KLHL12""","""PNKP""","""Glu326Lys""","""976G>A""","""ENSG00000039650""","""CEGS2""","""1_Pathogenic""",null,2.0,"""PNKP_Glu326Lys"""
100066360,14966,218521,"""VUSAPWT1B2""",0.875,1,1,1,1.0,1.0,"""KRT27""","""KRT8""","""Gly434Ser""","""1300G>A""","""ENSG00000170421""","""CEGS2""","""2_Benign""",null,2.0,"""KRT8_Gly434Ser"""
70539,5424,0,"""VUSAPWT1B2""",0.5,1,1,1,null,null,"""PXN""","""BLK""",null,null,"""ENSG00000136573""",null,null,null,null,"""BLK"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2500,2713,200832,"""VUSAPWT1B1""",0.125,1,1,1,1.0,1.0,"""ZNF212""","""MLH1""","""Gly147Arg""","""439G>C""","""ENSG00000076242""","""CEGS2""","""3_Conflicting""",null,1.0,"""MLH1_Gly147Arg"""
3562,5066,201831,"""VUSAPWT6B1""",0.0,6,1,1,1.0,2.0,"""PDLIM7""","""BAG3""","""Leu462Pro""","""1385T>C""","""ENSG00000151929""","""CEGS2""","""1_Pathogenic""",null,1.0,"""BAG3_Leu462Pro"""
6869,12043,0,"""VUSAPWT1B2""",0.25,1,1,1,null,null,"""LHX4""","""ORC6""",null,null,"""ENSG00000091651""",null,null,null,null,"""ORC6"""


In [4]:
scores = pl.read_csv(f"{outputs_dir}/4_normalized_consensus_scores.csv")

# keep only scores where there is at least one matched reference-variant set
orfs_with_alleles = scores.filter(pl.col("db_mut_id") != 0).select("db_orf_id").unique().to_series().to_list()
orfs = scores.filter(pl.col("db_mut_id") == 0).select("db_orf_id").unique().to_series().to_list()

complete_orf_pairs = [i for i in orfs if i in orfs_with_alleles]
scores = scores.filter(pl.col("db_orf_id").is_in(complete_orf_pairs))
genes = scores.select("db_orf_id").to_series().unique().to_list()

# print(scores)

In [5]:
# Create profiles for each profiled gene

profiles = {}
shape_summary = []
for gene in genes:
    gene_scores = scores.filter(pl.col("db_orf_id") == gene)
    profile = gene_scores.pivot(
                values="normalized_score",
                index="ad_orf_id",
                on="db_mut_id"
            )
    profiles[gene] = profile

    shape_summary.append({
        "gene": gene,
        "n_interactors": profile.height,
        "n_alleles": profile.width
    })

    profile.write_csv(f"{outputs_dir}/edgotype_profiles/orf_id_{gene}.csv")

shape_summary = pl.DataFrame(shape_summary)

In [6]:
## Add a perturbation category for each edge to use for the N2H validation

# Extract WT scores
wt_scores = scores.filter(pl.col("db_mut_id") == 0).select(["db_orf_id", "ad_orf_id", "normalized_score"])

# Merge WT scores back onto the main df
scores_with_wt = scores.join(wt_scores, on=["db_orf_id", "ad_orf_id"], how="left", suffix="_wt")

# Compute allele/WT ratio
scores_with_wt = scores_with_wt.with_columns(
    (pl.col("normalized_score") / pl.col("normalized_score_wt")).alias("ratio")
)

# Determine if the allele is perturbed - binary output
threshold = 0.5
scores_with_wt = scores_with_wt.with_columns(
    ((pl.col("db_mut_id") != 0) & (pl.col("ratio") < threshold)).alias("perturbed")
)

# Build perturbation_classes expression
bins = [
    (1.0, float("inf"), "increased edge"),
    (0.8, 1.0, "similar"),
    (0.6, 0.8, "slightly perturbed"),
    (0.4, 0.6, "perturbed"),
    (0.2, 0.4, "highly perturbed"),
    (0.0, 0.2, "very highly perturbed"),
]

perturbation_expr = pl.when(pl.col("db_mut_id") == 0).then(pl.lit("WT"))

perturbation_expr = perturbation_expr.when(pl.col("normalized_score_wt") == 0.0).then(pl.lit("WT edge lost"))

perturbation_expr = perturbation_expr.when(pl.col("ratio") == 0.0).then(pl.lit("edge lost"))

perturbation_expr = perturbation_expr.when(pl.col("ratio") == 1.0).then(pl.lit("WT-like"))

for lower, upper, label in bins:
    perturbation_expr = perturbation_expr.when((pl.col("ratio") > lower) & (pl.col("ratio") <= upper)).then(pl.lit(label))


perturbation_expr = perturbation_expr.otherwise(pl.lit("unclassified"))




scores_with_wt = scores_with_wt.with_columns(
    perturbation_expr.alias("perturbation_classes")
)

scores_with_perturbed = scores_with_wt.select("*")


pert2 = (pl.when(pl.col("db_mut_id") == 0).then(pl.lit('WT')))

pert2 = pert2.when(pl.col("normalized_score_wt") == 0.0).then(pl.lit("WT edge lost"))

pert2 = pert2.when(pl.col("ratio") == 0.0).then(pl.lit("edge lost"))

pert2 = pert2.when(pl.col("ratio") == 1.0).then(pl.lit("WT-like"))

bins_2 = [
    (float("inf"), 1.0, "increased edge"),
    (1, 0.7, 'slightly perturbed'),
    (0.7, 0.3, 'perturbed'),
    (0.3, 0, 'highly perturbed')
]

for upper, lower, label in bins_2:
    pert2 = pert2.when(
        (pl.col("ratio") <= upper) &
        (pl.col("ratio") >  lower)
    ).then(pl.lit(label))

pert2 = pert2.otherwise(pl.lit("unclassified"))

scores_with_perturbed = scores_with_wt.with_columns(
    perturbation_classes_2 = pert2.alias("perturbation_classes_2")
)



cols_to_keep = ['ad_orf_id',
 'db_orf_id',
 'db_mut_id',
 'normalized_score',
 'ad_symbol',
 'symbol',
 'clinvar_clnsig_clean',
 'normalized_score_wt',
 'ratio',
 'perturbed',
 'perturbation_classes',
 "perturbation_classes_2"]

scores_with_perturbed = scores_with_perturbed.select(cols_to_keep)

scores_with_perturbed.write_csv(f"{outputs_dir}/7_edge_perturbation_categorical.csv")

## Examine profile characteristics

In [7]:
# Number of profiled alleles with more than 5 interactors
hq_orfs = shape_summary.filter(pl.col("n_interactors") >= 5)
num_orfs = hq_orfs.shape[0]
print(num_orfs)

print(hq_orfs.select(pl.col("n_alleles")).sum() - num_orfs)

26
shape: (1, 1)
┌───────────┐
│ n_alleles │
│ ---       │
│ i64       │
╞═══════════╡
│ 258       │
└───────────┘


In [8]:
# Total number of profiled alleles
prof_genes = shape_summary.shape[0]
print(prof_genes)

print(shape_summary.select(pl.col("n_alleles")).sum() - prof_genes)

84
shape: (1, 1)
┌───────────┐
│ n_alleles │
│ ---       │
│ i64       │
╞═══════════╡
│ 659       │
└───────────┘


# Automated

In [11]:
scores = pl.read_csv(f"{outputs_dir}/4_automated_normalized_consensus_scores.csv")
scores = scores.drop_nulls(subset=["normalized_score"])

# keep only scores where there is at least one matched reference-variant set
orfs_with_alleles = scores.filter(pl.col("db_mut_id") != 0).select("db_orf_id").unique().to_series().to_list()
orfs = scores.filter(pl.col("db_mut_id") == 0).select("db_orf_id").unique().to_series().to_list()

complete_orf_pairs = [i for i in orfs if i in orfs_with_alleles]
scores = scores.filter(pl.col("db_orf_id").is_in(complete_orf_pairs))
genes = scores.select("db_orf_id").to_series().unique().to_list()

scores

ad_orf_id,db_orf_id,db_mut_id,retest_batch,adjusted_score,wt_summed_score,normalized_score,assay_id,seq_ad,seq_db,db_sequenced,db_sequence_confirmation_class,ad_symbol,symbol,aa_change,nt_change,ensembl_gene_id,collection,clinvar_clnsig_clean,gnomad_af,StarStatus,allele_0
i64,i64,i64,str,f64,f64,f64,i64,i64,i64,f64,f64,str,str,str,str,str,str,str,f64,f64,str
9625,71001,205677,"""VUSAPWT1B1""",1801.907129,1606.8288,1.121406,1,1,1,1.0,1.0,"""MAGED1""","""UBQLN2""","""Pro525Ser""","""1573C>T""","""ENSG00000188021""","""CEGS2""","""3_Conflicting""",0.000119,1.0,"""UBQLN2_Pro525Ser"""
4946,4062,201423,"""VUSAPWT1B2""",5.852614,1081.278893,0.005413,1,1,1,1.0,1.0,"""PTPRN""","""PLP1""","""Gly74Arg""","""220G>C""","""ENSG00000123560""","""CEGS2""",null,null,null,"""PLP1_Gly74Arg"""
100066360,4870,215298,"""VUSAPWT1B2""",1010.094163,2093.36937,0.482521,1,1,1,1.0,1.0,"""KRT27""","""KRT6A""","""Glu472Asp""","""1416G>C""","""ENSG00000205420""","""CEGS2""","""5_Others""",null,null,"""KRT6A_Glu472Asp"""
2831,6408,0,"""VUSAPWT6B1""",1055.091212,1055.091212,1.0,6,1,1,null,null,"""TMEM80""","""IER3IP1""",null,null,"""ENSG00000134049""",null,null,null,null,"""IER3IP1"""
9625,5066,201830,"""VUSAPWT1B2""",1146.532253,698.503619,1.641412,1,1,1,1.0,1.0,"""MAGED1""","""BAG3""","""Arg258Trp""","""772C>T""","""ENSG00000151929""","""CEGS2""","""2_Benign""",0.0004399,2.0,"""BAG3_Arg258Trp"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
824,3535,201190,"""VUSAPWT1B1""",0.0,298.664157,0.0,1,1,1,1.0,1.0,"""MEOX2""","""EMD""","""Met1Val""","""1A>G""","""ENSG00000102119""","""CEGS2""","""1_Pathogenic""",9.5000e-7,2.0,"""EMD_Met1Val"""
71959,70497,3888,"""VUSAPWT1B1""",15.991832,289.025168,0.05533,1,1,1,1.0,1.0,"""KRTAP1-1""","""AGXT""","""Glu371Lys""","""1111G>A""","""ENSG00000172482""","""Edgotyping3""","""3_Conflicting""",0.000141,1.0,"""AGXT_Glu371Lys"""
11830,7675,202585,"""VUSAPWT2B1""",3.717974,8.111111,0.45838,2,1,1,1.0,1.0,"""KPNA2""","""NMNAT1""","""Glu257Lys""","""769G>A""","""ENSG00000173614""","""CEGS2""","""1_Pathogenic""",0.0010295,2.0,"""NMNAT1_Glu257Lys"""


In [12]:
# Create profiles for each profiled gene

profiles = {}
shape_summary = []
for gene in genes:
    gene_scores = scores.filter(pl.col("db_orf_id") == gene)
    profile = gene_scores.pivot(
                values="normalized_score",
                index="ad_orf_id",
                on="db_mut_id"
            )
    profiles[gene] = profile

    shape_summary.append({
        "gene": gene,
        "n_interactors": profile.height,
        "n_alleles": profile.width
    })

    profile.write_csv(f"{outputs_dir}/edgotype_profiles/orf_id_{gene}_auto_normalized.csv")

shape_summary = pl.DataFrame(shape_summary)

In [13]:
# Create profiles for each profiled gene

profiles = {}
shape_summary = []
for gene in genes:
    gene_scores = scores.filter(pl.col("db_orf_id") == gene)
    profile = gene_scores.pivot(
                values="adjusted_score",
                index="ad_orf_id",
                on="db_mut_id"
            )
    profiles[gene] = profile

    shape_summary.append({
        "gene": gene,
        "n_interactors": profile.height,
        "n_alleles": profile.width
    })

    profile.write_csv(f"{outputs_dir}/edgotype_profiles/orf_id_{gene}_auto_summed.csv")

shape_summary = pl.DataFrame(shape_summary)